# استخراج مفاتيح موديلات الهواتف


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
print(project_root)

c:\Users\user\DEPI-GRADUATION-PROJECT


In [ ]:
import pandas as pd

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
sys.path.insert(0, str(project_root))

## 1. تحميل بيانات التدريب


In [ ]:
import zipfile
zip_path = "/content/project_splits2.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/project_splits2")

In [ ]:
import os

for root, dirs, files in os.walk("/content/project_splits2"):
    for file in files:
        print(os.path.join(root, file))

/content/project_splits2/stage1_val_df.csv
/content/project_splits2/stage1_df.csv
/content/project_splits2/stage1_test_df.csv
/content/project_splits2/project_splits2/stage1_val_df.csv
/content/project_splits2/project_splits2/stage1_df.csv
/content/project_splits2/project_splits2/stage1_test_df.csv
/content/project_splits2/project_splits2/data/splits/validation.csv
/content/project_splits2/project_splits2/data/processed/train_set.csv
/content/project_splits2/project_splits2/data/processed/frozen_test_set.csv


In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/project_splits2/stage1_df.csv")

val_df = pd.read_csv("/content/project_splits2/stage1_val_df.csv")

stage1_df = train_df.copy()
stage1_val_df = val_df.copy()


In [ ]:
print(stage1_df.shape)
print(stage1_val_df.shape)


(5165, 53)
(1252, 53)


In [ ]:
stage1_df[["title", "brand"]].head()

,title,brand
0,ايفون 13 iPhone,apple
1,Honor Magic V5,honor
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,oppo
3,Honor Robot Phone 512G 12G ram,honor
4,شاومي 17 الترا xiaomi 17 ultra,xiaomi


## 2. تحميل مجموعة الجولد


In [ ]:
#gold_path = Path("../data/labels/gold_set_Aya.csv")
gold_path = Path("/content/gold_set_Aya.csv")

print( gold_path.exists())

gold_df = pd.read_csv(gold_path)
print( gold_df.shape)
gold_df[["title", "brand", "gold_model_key"]].head()

True
(200, 8)


,title,brand,gold_model_key
0,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple,apple iphone air
1,iPhone 12 pro ايفون ١٢ برو,apple,apple iphone 12 pro
2,آبل - آيفون 13,apple,apple iphone 13
3,iPhone 16 Pro Max,apple,apple iphone 16 pro max
4,آبل - آيفون 13 برو ماكس,apple,apple iphone 13 pro max


In [ ]:
print(gold_df.columns.tolist())


['ad_id', 'title', 'brand', 'gold_model_key', 'review_notes', 'predicted_model_key', 'predicted_match_rule', 'confidence']


In [ ]:
gold_df["gold_model_key"].notna().sum() # sum of NaN

np.int64(200)

### مراجعة قاعدة Samsung من الدفتر السابق


In [ ]:
import sys
from pathlib import Path

project_path = Path("/content/DEPI-GRADUATION-PROJECT")
sys.path.insert(0, str(project_path))

from src.preprocessing.model_keys import extract_brand_model

print(extract_brand_model("Samsung Galaxy A54", "samsung"))

samsung a54


#### فحص نتائج Samsung


In [ ]:
samsung_gold = gold_df.loc[
    gold_df["brand"] == "samsung"
].copy()

In [ ]:
samsung_gold["my_prediction"]=samsung_gold["title"].apply(
    lambda title : extract_brand_model(title,'samsung')
                                                             )

In [ ]:
display(
    samsung_gold[
        ["title", "gold_model_key", "my_prediction"]
    ].head(10)
)

,title,gold_model_key,my_prediction
91,Samsung S26 Ultra,samsung s26 ultra,samsung s26 ultra
92,Samsung A52,samsung a52,samsung a52
93,سامسونج ايه 30,samsung a30,samsung a30
94,Galaxy M55 5G 256,samsung m55,samsung m55
95,سامسونج اس 25 الترا,samsung s25 ultra,samsung s25 ultra
96,سامسونج ايه 30,samsung a30,samsung a30
97,Samsung galagxy s25 ultra,samsung s25 ultra,samsung s25 ultra
98,Galaxy Z Fold 7 dual sim 512G Blue جديد متبرشم,samsung z fold7,samsung z fold7
99,سامسونج ايه 37,samsung a37,samsung a37
100,سامسونج ايه 23,samsung a23,samsung a23


In [ ]:
"""Erros"""


samsung_errors = samsung_gold.loc[
    samsung_gold["my_prediction"] != samsung_gold["gold_model_key"],
    ["title", "gold_model_key", "my_prediction"]
]

In [ ]:
print(f"Errors: {len(samsung_errors)}") # عدد الاختلافات هنا بيساوي = 1

Errors: 1


In [ ]:
display(samsung_errors)

,title,gold_model_key,my_prediction
112,سامسونج موديل اخر,unknown,samsung other


In [ ]:
stage1_df['brand'].value_counts()

,count
brand,
apple,2462
samsung,749
xiaomi,426
oppo,361
honor,224
realme,178
infinix,138
vivo,128
huawei,99


#### فحص نتائج Apple


In [ ]:
import importlib
from src.preprocessing import model_keys

importlib.reload(model_keys)

gold_df["my_prediction"] = gold_df.apply(
    lambda row: model_keys.predict_model_key(row["title"], row["brand"]),
    axis=1
)

display(gold_df[["brand", "title", "gold_model_key", "my_prediction"]].head(10))

,brand,title,gold_model_key,my_prediction
0,apple,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple iphone air,apple iphone air
1,apple,iPhone 12 pro ايفون ١٢ برو,apple iphone 12 pro,apple iphone 12 pro
2,apple,آبل - آيفون 13,apple iphone 13,apple iphone 13
3,apple,iPhone 16 Pro Max,apple iphone 16 pro max,apple iphone 16 pro max
4,apple,آبل - آيفون 13 برو ماكس,apple iphone 13 pro max,apple iphone 13 pro max
5,apple,آبل - آيفون 13,apple iphone 13,apple iphone 13
6,apple,Apple - iPhone 13,apple iphone 13,apple iphone 13
7,apple,Apple - iPhone 15 plus,apple iphone 15 plus,apple iphone 15 plus
8,apple,آبل - آيفون 17 برو ماكس هاي كوبي ١٠٠٠ جيجا,apple iphone 17 pro max,apple iphone 17 pro max
9,apple,آبل - آيفون 17 برو ماكس فرست هاي كوبي,apple iphone 17 pro max,apple iphone 17 pro max


In [ ]:
apple_errors = gold_df.loc[
    (gold_df["brand"] == "apple")
    & (gold_df["my_prediction"] != gold_df["gold_model_key"]),
    ["title", "gold_model_key", "my_prediction"]
]

In [ ]:
apple_errors

,title,gold_model_key,my_prediction
0,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple iphone air,unknown
53,آبل - آيفون 4 اس,apple iphone 4s,apple iphone 4
62,آبل - آيفون 13 ميني,apple iphone 13 mini,apple iphone 13
64,لو عندك ايفون 14 او 15 او 16 او 17 وعاوز تبيعو...,unknown,apple iphone 14


#### فحص نتائج Xiaomi


In [ ]:
xiaomi_gold = gold_df.loc[
    gold_df["brand"] == "xiaomi"
].copy()

xiaomi_errors = xiaomi_gold.loc[
    xiaomi_gold["my_prediction"] != xiaomi_gold["gold_model_key"],
    ["title", "gold_model_key", "my_prediction"]
]


In [ ]:
len(xiaomi_errors)

2

In [ ]:
xiaomi_errors

,title,gold_model_key,my_prediction
128,تليفون Poco x3 nfc 8/128,xiaomi poco x3 nfc,xiaomi poco x3
140,Poco x8 pro max,xiaomi poco x8 pro max,xiaomi poco x8 pro


#### فحص نتائج OPPO


In [ ]:
oppo_gold = gold_df.loc[
    gold_df["brand"] == "oppo"
].copy()

oppo_errors = oppo_gold.loc[
    oppo_gold["my_prediction"] != oppo_gold["gold_model_key"],
    ["title", "gold_model_key", "my_prediction"]
]



In [ ]:
oppo_errors

,title,gold_model_key,my_prediction


### فحص بقية الشركات


In [ ]:
def review_brand(brand_name):
    brand_gold = gold_df.loc[
        gold_df["brand"] == brand_name
    ].copy()

    brand_errors = brand_gold.loc[
        brand_gold["my_prediction"] != brand_gold["gold_model_key"],
        ["title", "gold_model_key", "my_prediction"]
    ]

    print("Brand:", brand_name)
    print("len:", len(brand_gold))
    print("Errors:", len(brand_errors))
    display(brand_errors)

In [ ]:
review_brand("realme")

Brand: realme
len: 7
Errors: 1


,title,gold_model_key,my_prediction
167,realme c85 pro,realme c85 pro,realme c85


In [ ]:
review_brand("honor")

Brand: honor
len: 6
Errors: 1


,title,gold_model_key,my_prediction
173,هونر اكس 9 سى سمارت,honor x9c smart,honor x9c


In [ ]:
review_brand("infinix")

Brand: infinix
len: 5
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("vivo")

Brand: vivo
len: 4
Errors: 2


,title,gold_model_key,my_prediction
183,Vivo V60 lite 5g,vivo v60 lite 5g,vivo v60 lite
185,vivo v40 ite,vivo v40 lite,vivo v40


In [ ]:
review_brand("huawei")

Brand: huawei
len: 4
Errors: 1


,title,gold_model_key,my_prediction
187,هواوى نوفا ماكس استعمال 10 ايام بالفاتورة والض...,huawei nova max,unknown


In [ ]:
review_brand("motorola")

Brand: motorola
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("oneplus")

Brand: oneplus
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("oneplus")

Brand: oneplus
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("zte")

Brand: zte
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("google")

Brand: google
len: 1
Errors: 1


,title,gold_model_key,my_prediction
193,Google Pixel 11 Pro Fold 16g 256g Black New,google pixel 11 pro fold,google 11 pro fold


In [ ]:
review_brand("tecno")

Brand: tecno
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("sony")

Brand: sony
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("itel")

Brand: itel
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("nokia")

Brand: nokia
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("nokia")

Brand: nokia
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("lenovo")

Brand: lenovo
len: 1
Errors: 0


,title,gold_model_key,my_prediction


In [ ]:
review_brand("lg")

Brand: lg
len: 1
Errors: 0


,title,gold_model_key,my_prediction


## 3. تحديث القواعد بناءً على الأخطاء


In [ ]:
"""
 here iam fix orignal file "model_keys.py"
"""

'\n here iam fix orignal file "model_keys.py"\n'

In [ ]:
import importlib
from src.preprocessing import model_keys

importlib.reload(model_keys)

gold_df["my_prediction"] = gold_df.apply(
    lambda row: model_keys.predict_model_key(row["title"], row["brand"]),
    axis=1
)

In [ ]:
gold_df["my_prediction"]

,my_prediction
0,apple iphone air
1,apple iphone 12 pro
2,apple iphone 13
3,apple iphone 16 pro max
4,apple iphone 13 pro max
...,...
195,sony st15i
196,itel a90
197,nokia x2
198,lenovo k12 note


In [ ]:
errors = gold_df.loc[
    gold_df["my_prediction"] != gold_df["gold_model_key"],
    ["brand", "title", "gold_model_key", "my_prediction"]
]

print("Errors:", len(errors))
display(errors)

Errors: 5


,brand,title,gold_model_key,my_prediction
154,NaN,Mi 17 Ultra dual sim 512/1TB Black جديد متبرشم...,xiaomi 17 ultra,unknown
156,NaN,Turbo 4 pro كسر زيرو,xiaomi redmi turbo 4 pro,unknown
157,NaN,لبيع ميت تن لايت,huawei mate 10 lite,unknown
159,NaN,17 Pro Max,apple iphone 17 pro max,unknown
160,NaN,Mate 30 pro 5g,huawei mate 30 pro 5g,unknown


In [ ]:
print(float(len(errors)/len(gold_df["my_prediction"])) *100)

2.5


## 4. تقييم القواعد على مجموعة الجولد


In [ ]:
import importlib
from src.preprocessing import model_keys

In [ ]:
gold_pairs = gold_df.apply(
    lambda row: model_keys.predict_model_key(
        row["title"],
        row["brand"],
        with_rule=True
    ),
    axis=1
)

In [ ]:
gold_df[["my_prediction", "match_rule"]] = pd.DataFrame(
    gold_pairs.tolist(),
    index=gold_df.index
)

In [ ]:
display(gold_df[["brand", "title", "my_prediction", "match_rule"]].head(10)) # بعد ما عدلت في الملف الاخر و ضيفت  enumrate علشان اعرف انهب case هي الي عامله المشكله

,brand,title,my_prediction,match_rule
0,apple,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple iphone air,apple:special_case_01
1,apple,iPhone 12 pro ايفون ١٢ برو,apple iphone 12 pro,apple:model_pattern_06
2,apple,آبل - آيفون 13,apple iphone 13,apple:fallback_pattern
3,apple,iPhone 16 Pro Max,apple iphone 16 pro max,apple:model_pattern_01
4,apple,آبل - آيفون 13 برو ماكس,apple iphone 13 pro max,apple:model_pattern_02
5,apple,آبل - آيفون 13,apple iphone 13,apple:fallback_pattern
6,apple,Apple - iPhone 13,apple iphone 13,apple:numeric_pattern
7,apple,Apple - iPhone 15 plus,apple iphone 15 plus,apple:model_pattern_03
8,apple,آبل - آيفون 17 برو ماكس هاي كوبي ١٠٠٠ جيجا,apple iphone 17 pro max,apple:model_pattern_02
9,apple,آبل - آيفون 17 برو ماكس فرست هاي كوبي,apple iphone 17 pro max,apple:model_pattern_02


In [ ]:
scored_gold = gold_df.loc[
    gold_df["my_prediction"] != "unknown"
].copy()

In [ ]:
scored_gold["is_correct"] = (
    scored_gold["my_prediction"] == scored_gold["gold_model_key"]
)

In [ ]:
rule_stats = scored_gold.groupby("match_rule").agg(
    support=("is_correct", "size"),
    correct=("is_correct", "sum")
)

In [ ]:
rule_stats["confidence"] = (
    rule_stats["correct"] / rule_stats["support"]
).where(rule_stats["support"] >= 5)

In [ ]:
gold_df["confidence"] = gold_df["match_rule"].map(
    rule_stats["confidence"]
)

In [ ]:
gold_df["is_correct"] = (
    gold_df["my_prediction"] == gold_df["gold_model_key"]
)

display(
    gold_df[
        ["gold_model_key", "my_prediction", "match_rule", "is_correct"]
    ].head(10)
)

,gold_model_key,my_prediction,match_rule,is_correct
0,apple iphone air,apple iphone air,apple:special_case_01,True
1,apple iphone 12 pro,apple iphone 12 pro,apple:model_pattern_06,True
2,apple iphone 13,apple iphone 13,apple:fallback_pattern,True
3,apple iphone 16 pro max,apple iphone 16 pro max,apple:model_pattern_01,True
4,apple iphone 13 pro max,apple iphone 13 pro max,apple:model_pattern_02,True
5,apple iphone 13,apple iphone 13,apple:fallback_pattern,True
6,apple iphone 13,apple iphone 13,apple:numeric_pattern,True
7,apple iphone 15 plus,apple iphone 15 plus,apple:model_pattern_03,True
8,apple iphone 17 pro max,apple iphone 17 pro max,apple:model_pattern_02,True
9,apple iphone 17 pro max,apple iphone 17 pro max,apple:model_pattern_02,True


In [ ]:
scored_gold = gold_df.loc[
    gold_df["my_prediction"] != "unknown"
].copy()

rule_stats = scored_gold.groupby("match_rule").agg(
    support=("is_correct", "size"),
    correct=("is_correct", "sum")
)

display(rule_stats)

,support,correct
match_rule,,
apple:fallback_pattern,14,14
apple:inferred_17_pro_max,1,1
apple:model_pattern_01,13,13
apple:model_pattern_02,23,23
apple:model_pattern_03,1,1
...,...,...
xiaomi:rule_06,3,3
xiaomi:rule_08,3,3
xiaomi:rule_15,1,1


In [ ]:
rule_stats["confidence"] = (
    rule_stats["correct"] / rule_stats["support"]
).where(rule_stats["support"] >= 5)

gold_df["confidence"] = gold_df["match_rule"].map(
    rule_stats["confidence"]
)

In [ ]:
rule_stats["confidence"]

,confidence
match_rule,
apple:fallback_pattern,1.0
apple:inferred_17_pro_max,NaN
apple:model_pattern_01,1.0
apple:model_pattern_02,1.0
apple:model_pattern_03,NaN
...,...
xiaomi:rule_06,NaN
xiaomi:rule_08,NaN
xiaomi:rule_15,NaN


In [ ]:
gold_df["confidence"]

,confidence
0,NaN
1,1.0
2,1.0
3,1.0
4,1.0
...,...
195,NaN
196,NaN
197,NaN
198,NaN


## 5. تطبيق القواعد على بيانات التدريب


In [ ]:
importlib.reload(model_keys)

<module 'src.preprocessing.model_keys' from '/content/DEPI-GRADUATION-PROJECT/src/preprocessing/model_keys.py'>

In [ ]:
train_pairs = stage1_df.apply(
    lambda row: model_keys.predict_model_key(
        row["title"],
        row["brand"],
        with_rule=True
    ),
    axis=1
)


In [ ]:
train_predictions = pd.DataFrame(
    train_pairs.tolist(),
    columns=["model_key", "match_rule"],
    index=stage1_df.index
)

stage1_train_models = stage1_df.copy()
stage1_train_models[["model_key", "match_rule"]] = train_predictions
stage1_train_models["confidence"] = stage1_train_models["match_rule"].map(
    rule_stats["confidence"]
)

display(
    stage1_train_models[
        ["brand", "title", "model_key", "match_rule", "confidence"]
    ].head(10)
)

,brand,title,model_key,match_rule,confidence
0,apple,ايفون 13 iPhone,apple iphone 13,apple:numeric_pattern,1.0
1,honor,Honor Magic V5,honor magic v5,honor:rule_05,NaN
2,oppo,OPPO Find X9 Pro ضمان محلي للبيع او البدل,oppo find x9 pro,oppo:rule_03,NaN
3,honor,Honor Robot Phone 512G 12G ram,honor robot,honor:rule_11,NaN
4,xiaomi,شاومي 17 الترا xiaomi 17 ultra,xiaomi 17 ultra,xiaomi:rule_17,NaN
5,apple,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,apple iphone 15 pro,apple:model_pattern_06,1.0
6,apple,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,apple iphone 11,apple:numeric_pattern,1.0
7,apple,آبل - آيفون 15 برو ماكس,apple iphone 15 pro max,apple:model_pattern_02,1.0
8,huawei,Huawei nova 15 max,huawei nova 15max,huawei:rule_09,NaN
9,apple,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iPh...,apple iphone 16e,apple:numeric_pattern,1.0


## 6. تطبيق القواعد على بيانات validation


In [ ]:
train_pairs = stage1_val_df.apply(
    lambda row: model_keys.predict_model_key(
        row["title"],
        row["brand"],
        with_rule=True
    ),
    axis=1
)


In [ ]:
train_predictions = pd.DataFrame(
    train_pairs.tolist(),
    columns=["model_key", "match_rule"],
    index=stage1_val_df.index
)

stage1_val_models = stage1_val_df.copy()
stage1_val_models[["model_key", "match_rule"]] = train_predictions
stage1_val_models["confidence"] = stage1_val_models["match_rule"].map(
    rule_stats["confidence"]
)

display(
    stage1_val_models[
        ["brand", "title", "model_key", "match_rule", "confidence"]
    ].head(10)
)

,brand,title,model_key,match_rule,confidence
0,honor,Honor 500 pro,honor 500 pro,honor:rule_08,NaN
1,apple,آبل - آيفون 13 برو ماكس,apple iphone 13 pro max,apple:model_pattern_02,1.0
2,samsung,سامسونج اس 24 الترا,samsung s24 ultra,samsung:rule_14,1.0
3,honor,Honor win rt,honor win rt,honor:rule_09,NaN
4,apple,Iphone 13 pro max,apple iphone 13 pro max,apple:model_pattern_01,1.0
5,samsung,سامسونج اس 24 الترا,samsung s24 ultra,samsung:rule_14,1.0
6,apple,Apple - iPhone 17 Pro Max - Battery 98%,apple iphone 17 pro max,apple:special_case_09,NaN
7,samsung,Galaxy Z TriFold,samsung z trifold,samsung:rule_01,NaN
8,vivo,Vivo V60,vivo v60,vivo:rule_06,NaN
9,samsung,سامسونج اس 25 الترا,samsung s25 ultra,samsung:rule_14,1.0


### حفظ ملف النتائج


In [ ]:
#stage1_df.to_csv(
#    "../data/processed/stage1/stage1_model_keys.csv",
#    index=False
#)